# Lab 5 — Initialization Strategies and Vanishing Gradients

## How initialization affects deep neural networks

### Description

In this lab, you will explore how different weight initialization methods influence the training of deep neural networks.

You will see that a bad initialization can cause problems such as:

* Vanishing gradients (training becomes extremely slow)
* Exploding gradients (instability and divergence)
* Activation saturation (neurons stop learning)
* Dead ReLU units

To understand these effects, we will experiment with:

1. Small 2D datasets (circles, moons)
2. A more realistic dataset: cats vs non-cats

You will compare several initialization schemes:

* Bad initialization (intentionally chosen to be poor)
* Xavier initialization
* He initialization

You will also observe how these schemes interact with different activation functions:

* tanh
* sigmoid
* ReLU

---

### Objectives

By the end of the lab, you should be able to:

#### Conceptual objectives

* Explain what causes vanishing and exploding gradients.
* Explain the role of Xavier and He initialization.
* Describe how initialization interacts with tanh, sigmoid and ReLU.

#### Practical objectives

* Train deep networks using different initialization schemes.
* Visualize gradient evolution across layers.
* Measure and plot activation saturation during training.
* Compare initial vs final weight distributions.
* Run and interpret the same experiments on a realistic dataset (cats vs non-cats).

---

### What you will produce

During this lab, you will generate:

* Loss curves
* Gradient evolution plots (per layer)
* Saturation curves
* Weight evolution histograms
* Short written conclusions based on your experiments


In [ ]:
# import packages
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import h5py
import copy
from sklearn.datasets import make_circles, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

%matplotlib inline
plt.rcParams['figure.figsize'] = (7.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Loader for the classic train_catvnoncat.h5 dataset

In [ ]:
def load_cats_h5(train_path, test_path):
    """
    Load the classic 'cats vs non-cats' dataset stored in H5 files.

    train_path: path to train_catvnoncat.h5
    test_path:  path to test_catvnoncat.h5

    Returns:
        X_train_orig: (m_train, n_H, n_W, n_C)
        y_train_orig: (m_train,)
        X_test_orig:  (m_test,  n_H, n_W, n_C)
        y_test_orig:  (m_test,)
    """
    with h5py.File(train_path, "r") as train_h5:
        X_train_orig = np.array(train_h5["train_set_x"][:])  # images
        y_train_orig = np.array(train_h5["train_set_y"][:])  # labels

    with h5py.File(test_path, "r") as test_h5:
        X_test_orig = np.array(test_h5["test_set_x"][:])
        y_test_orig = np.array(test_h5["test_set_y"][:])

    y_train_orig = y_train_orig.reshape(-1)  # (m_train,)
    y_test_orig = y_test_orig.reshape(-1)    # (m_test,)

    return X_train_orig, y_train_orig, X_test_orig, y_test_orig

# Flatten + optional PCA for the NN

This function prepares the data in the format suitable for your NN + gradient analysis framework.

In [ ]:
def preprocess_cats_dataset(X_train_orig, y_train_orig,
                            X_test_orig, y_test_orig,
                            use_pca=False,
                            pca_dim=100,
                            test_size=0.2,
                            random_state=0):
    """
    Preprocess cats vs non-cats images for a fully-connected network.

    Steps:
      - Flatten images to vectors
      - Normalize pixel values to [0,1]
      - Optionally apply PCA to reduce dimensionality
      - Split train into (train/test) for evaluation in this notebook

    Returns:
      X_train: (m_train, n_features)
      y_train: (m_train,)
      X_test:  (m_test,  n_features)
      y_test:  (m_test,)
    """
    # 1) Flatten and scale to [0,1]
    m_train = X_train_orig.shape[0]
    m_test  = X_test_orig.shape[0]

    X_train_flat = X_train_orig.reshape(m_train, -1).astype(np.float32) / 255.0
    X_test_flat  = X_test_orig.reshape(m_test,  -1).astype(np.float32) / 255.0

    y_train = y_train_orig.astype(int)
    y_test  = y_test_orig.astype(int)

    # 2) Optional PCA (fit on training data only)
    if use_pca:
        print(f"Applying PCA to reduce dimensionality to {pca_dim}...")
        pca = PCA(n_components=pca_dim, random_state=random_state)
        X_train_flat = pca.fit_transform(X_train_flat)
        X_test_flat  = pca.transform(X_test_flat)
        print("Explained variance ratio (sum):", np.sum(pca.explained_variance_ratio_))

    # 3) Optional: re-split train into train/validation; here we keep a simple train/test split
    # If you want an internal split from the original train set:
    # X_train_flat, X_test_flat, y_train, y_test = train_test_split(
    #     X_train_flat, y_train, test_size=test_size, random_state=random_state, stratify=y_train
    # )

    print(f"X_train shape: {X_train_flat.shape}, X_test shape: {X_test_flat.shape}")
    print(f"Class balance train: {np.bincount(y_train)}, test: {np.bincount(y_test)}")

    return X_train_flat, y_train, X_test_flat, y_test


# Unified loader: 2D (circles/moons) vs cats

This function lets you flip with one flag between the small 2D dataset (like Part 1) and the cats/non-cats dataset, but returns data formatted for the training code.

In [ ]:
def load_dataset_flexible(dataset="circles",
                          cats_train_path=None,
                          cats_test_path=None,
                          use_pca=False,
                          pca_dim=100,
                          random_state=0):
    """
    Flexible loader that returns (X_train, y_train, X_test, y_test)
    ready for the MLP + gradient framework.

    Shapes:
      X_train: (m_train, n_features)
      y_train: (m_train,)
      X_test:  (m_test,  n_features)
      y_test:  (m_test,)

    dataset:
      - "circles" or "moons": synthetic 2D datasets
      - "cats": uses H5-based cats vs non-cats dataset
    """
    if dataset in ["circles", "moons"]:
        N_SAMPLES = 20000
        TEST_SIZE = 0.2

        if dataset == "circles":
            X, y = make_circles(n_samples=N_SAMPLES, noise=0.2, factor=0.4, random_state=random_state)
        else:
            X, y = make_moons(n_samples=N_SAMPLES, noise=0.25, random_state=random_state)

        scaler = StandardScaler().fit(X)
        X_scaled = scaler.transform(X).astype(np.float32)
        y = y.astype(int)

        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=TEST_SIZE, random_state=random_state, stratify=y
        )

        print(f"[{dataset}] X_train: {X_train.shape}, X_test: {X_test.shape}")
        print(f"[{dataset}] Class balance train: {np.bincount(y_train)}, test: {np.bincount(y_test)}")

        return X_train, y_train, X_test, y_test

    elif dataset == "cats":
        assert cats_train_path is not None and cats_test_path is not None, \
            "Provide cats_train_path and cats_test_path for dataset='cats'."

        X_train_orig, y_train_orig, X_test_orig, y_test_orig = load_cats_h5(
            cats_train_path, cats_test_path
        )

        X_train, y_train, X_test, y_test = preprocess_cats_dataset(
            X_train_orig, y_train_orig,
            X_test_orig, y_test_orig,
            use_pca=use_pca,
            pca_dim=pca_dim,
            random_state=random_state,
        )

        return X_train, y_train, X_test, y_test

    else:
        raise ValueError(f"Unknown dataset: {dataset}. Choose 'circles', 'moons', or 'cats'.")


# To do:

In the following cells, functions to build a feed-forward DNN are provided.

Analyse these function and for each function complete the description.

In [ ]:
def get_activation(name):
    """Return (activation_function, activation_derivative) for the given name.
    Supported activations: "tanh", "sigmoid" (input clipped to [-40, 40] for stability), "relu".
    Both returned functions act element-wise on Z; f_prime(Z) is used in backprop (dZ = dA * f'(Z)).
    """
    name = name.lower()

    if name == "tanh":
        def f(z):
            return np.tanh(z)
        def f_prime(z):
            return 1 - np.tanh(z)**2
        return f, f_prime

    elif name == "sigmoid":
        def f(z):
            # Numerically stable sigmoid via input clipping
            zc = np.clip(z, -40.0, 40.0)
            return 1.0 / (1.0 + np.exp(-zc))
        def f_prime(z):
            a = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
            return a * (1.0 - a)
        return f, f_prime

    elif name == "relu":
        def f(z):
            return np.maximum(0, z)
        def f_prime(z):
            return (z > 0).astype(float)
        return f, f_prime

    else:
        raise ValueError(f"Unsupported activation: {name}. Choose among 'tanh', 'sigmoid', 'relu'.")


def forward_propagation_deep(X, parameters, activation="tanh"):
    """Forward pass for an L-layer fully-connected network.
    Hidden layers: Z = W @ A_prev + b, A = activation(Z); output layer: sigmoid (binary classification).
    X has shape (n_x, m). Returns AL (1, m) and caches {A0..AL, Z1..ZL, W1..WL, b1..bL, params} for backprop.
    """
    f_hidden, _ = get_activation(activation)
    f_sigmoid, _ = get_activation("sigmoid")

    caches = {}
    A = X
    caches["A0"] = A

    # Keep a light reference to parameters in cache (fallback for backprop)
    caches["params"] = parameters

    L = len([k for k in parameters.keys() if k.startswith("W")])

    for l in range(1, L + 1):
        W = parameters[f"W{l}"]
        b = parameters[f"b{l}"]
        Z = W @ A + b
        caches[f"Z{l}"] = Z
        caches[f"W{l}"] = W
        caches[f"b{l}"] = b

        if l < L:
            A = f_hidden(Z)
        else:
            A = f_sigmoid(Z)
        caches[f"A{l}"] = A

    AL = A  # shape (1, m)
    return AL, caches


def backward_propagation_deep(AL, Y, caches, activation="tanh"):
    """Backward pass (backpropagation) for the network built by forward_propagation_deep.
    Uses the sigmoid + binary cross-entropy shortcut dZL = AL - Y, then for each layer l (L..1):
    dW = dZ A_prev^T / m, db = mean(dZ), dZ_prev = (W^T dZ) * f'(Z_prev).
    Returns grads {dW1..dWL, db1..dbL, dZ1..dZL}.
    """
    if Y.ndim == 1:
        Y = Y.reshape(1, -1)
    m = Y.shape[1]

    # Number of layers
    L = len([k for k in caches.keys() if k.startswith("Z")])

    # Output layer gradient
    dZ = AL - Y

    grads = {f"dZ{L}": dZ}

    _, f_prime = get_activation(activation)

    for l in reversed(range(1, L + 1)):
        A_prev = caches[f"A{l-1}"]
        # Try to get W from caches; fallback to stored params
        W = caches.get(f"W{l}")
        if W is None:
            W = caches["params"][f"W{l}"]

        dW = (1.0 / m) * (dZ @ A_prev.T)
        db = (1.0 / m) * np.sum(dZ, axis=1, keepdims=True)

        grads[f"dW{l}"] = dW
        grads[f"db{l}"] = db

        if l > 1:
            dA_prev = W.T @ dZ
            Z_prev = caches[f"Z{l-1}"]
            dZ = dA_prev * f_prime(Z_prev)
            grads[f"dZ{l-1}"] = dZ

    return grads


def init_grad_history(parameters, track=("dZ", "dW", "db")):
    """Create an empty gradient history structure for all layers.

    Args:
        parameters: dict {W1, b1, ..., WL, bL} (used only to count the layers L)
        track: gradient kinds to record, e.g. ("dZ", "dW", "db")

    Returns:
        grad_hist: dict {kind: {f"{kind}{l}": [] for l in 1..L}}, e.g. grad_hist["dW"]["dW3"] = []
        
    """
    L = len([k for k in parameters.keys() if k.startswith("W")])
    grad_hist = {}
    for kind in track:
        grad_hist[kind] = {f"{kind}{l}": [] for l in range(1, L + 1)}
    return grad_hist


def update_grad_history(grad_hist, grads, metric="std"):
    """Append a scalar summary (metric) of each tracked gradient tensor to grad_hist (one value per iteration).

    Args:
        grad_hist: dict created by init_grad_history (modified in place)
        grads: gradients dict returned by backward_propagation_deep
        metric: reduction applied to each gradient tensor: "std", "l2", "mean_abs" or "max_abs"

    Notes:
        - std: np.std(tensor)
        - l2:  np.linalg.norm(tensor) / sqrt(tensor.size)  (scale-invariant)
        - mean_abs: np.mean(np.abs(tensor))
        - max_abs:  np.max(np.abs(tensor))
    """
    def reduce_metric(arr):
        if metric == "std":
            return float(np.std(arr))
        if metric == "l2":
            return float(np.linalg.norm(arr) / np.sqrt(arr.size))
        if metric == "mean_abs":
            return float(np.mean(np.abs(arr)))
        if metric == "max_abs":
            return float(np.max(np.abs(arr)))
        raise ValueError("Unknown metric: " + str(metric))

    for kind, per_layer in grad_hist.items():
        for key in per_layer.keys():  # e.g., "dZ3", "dW2", "db1"
            if key in grads:
                val = reduce_metric(grads[key])
                per_layer[key].append(val)

def init_saturation_history(parameters):
    """
    Create empty saturation history for each layer.
    Returns:
        sat_hist: dict like { "A1": [], "A2": [], ..., "AL": [] }
        where each entry will store a list of saturation fractions over iterations.
    """
    L = len([k for k in parameters.keys() if k.startswith("W")])
    sat_hist = {f"A{l}": [] for l in range(1, L + 1)}
    return sat_hist


def _saturation_fraction(A, activation, tanh_thr=0.99, sig_eps=1e-2):
    """
    Compute the fraction of 'saturated' units in activation matrix A
    depending on the activation type.

    A: np.ndarray (n_l, m)
    activation: "tanh", "sigmoid", "relu"
    """
    activation = activation.lower()
    A_flat = A.reshape(-1)

    if activation == "tanh":
        sat = np.abs(A_flat) > tanh_thr
    elif activation == "sigmoid":
        sat = (A_flat < sig_eps) | (A_flat > 1.0 - sig_eps)
    elif activation == "relu":
        sat = (A_flat <= 0.0)  # dead ReLU
    else:
        raise ValueError(f"Unsupported activation for saturation: {activation}")

    if A_flat.size == 0:
        return 0.0
    return float(np.mean(sat))


def update_saturation_history(sat_hist, caches, activation):
    """
    Append current saturation fraction for each layer based on caches.

    sat_hist: dict from init_saturation_history
    caches:  dict from forward_propagation_deep containing A1, A2, ..., AL
    activation: hidden activation ("tanh", "sigmoid", "relu")

    Notes:
      - The last layer in your network uses sigmoid for the output.
        Here we typically care about saturation in *hidden* layers, so
        you can choose to:
          - use the same 'activation' for all layers (simpler),
          - or treat the last layer separately if you wish.
    """
    for key in sat_hist.keys():  # e.g., "A1", "A2", ...
        if key in caches:
            A = caches[key]
            frac = _saturation_fraction(A, activation=activation)
            sat_hist[key].append(frac)


def train_deep_model(
    X,
    Y,
    layer_dims,
    init_scheme="bad",
    hidden_activation="relu",
    learning_rate=0.01,
    num_iterations=3000,
    print_every=200,
    track_grad_kinds=("dZ", "dW"),
    metric="std",
    snapshot_every=None,
    track_saturation=True,
):
    """
    Train a deep binary classifier with full-batch gradient descent.
    X: (n_x, m), Y: (m,) or (1, m). Steps per iteration: forward -> BCE loss -> backward -> W, b update,
    while recording gradient statistics per layer (grad_hist), saturation per layer (sat_hist)
    and optional activation/gradient snapshots.
    Returns: parameters, losses, grad_hist, initial_params, snapshots, sat_hist
    """
    # 1) Initialize parameters
    parameters = initialize_parameters_deep(
        layer_dims, scheme=init_scheme, activation=hidden_activation
    )
    initial_params = {k: v.copy() for k, v in parameters.items()}

    # 2) Trackers
    grad_hist = init_grad_history(parameters, track=track_grad_kinds)
    sat_hist = init_saturation_history(parameters) if track_saturation else None
    losses = []
    snapshots = []

    if Y.ndim == 1:
        Y = Y.reshape(1, -1)
    m = Y.shape[1]

    # 3) Training loop
    for it in range(num_iterations):
        # Forward
        AL, caches = forward_propagation_deep(X, parameters, activation=hidden_activation)

        # Binary cross-entropy loss (with clipping)
        eps = 1e-8
        ALc = np.clip(AL, eps, 1.0 - eps)
        loss = - (1.0 / m) * np.sum(Y * np.log(ALc) + (1 - Y) * np.log(1 - ALc))
        losses.append(float(loss))

        # Backward
        grads = backward_propagation_deep(AL, Y, caches, activation=hidden_activation)

        # Update params
        L = len([k for k in parameters if k.startswith("W")])
        for l in range(1, L + 1):
            parameters[f"W{l}"] -= learning_rate * grads[f"dW{l}"]
            parameters[f"b{l}"] -= learning_rate * grads[f"db{l}"]

        # Track gradient summaries
        update_grad_history(grad_hist, grads, metric=metric)

        # Track saturation summaries (per layer)
        if track_saturation:
            update_saturation_history(sat_hist, caches, activation=hidden_activation)

        # Optional snapshots for histograms
        if snapshot_every is not None and snapshot_every > 0 and (it % snapshot_every == 0 or it == num_iterations - 1):
            snap_caches = {k: v.copy() for k, v in caches.items() if k.startswith("A") or k.startswith("Z")}
            snap_grads = {k: v.copy() for k, v in grads.items() if k.startswith("dZ")}
            snapshots.append((it, snap_caches, snap_grads))

        if print_every and (it % print_every == 0 or it == num_iterations - 1):
            print(f"Iter {it:5d} | loss = {loss:.5f}")

    return parameters, losses, grad_hist, initial_params, snapshots, sat_hist


def _fan_in_out(n_prev, n_curr):
    """Utility: return (fan_in, fan_out)."""
    return n_prev, n_curr


def initialize_parameters_deep(layers_dims, scheme="bad", activation="tanh", seed=42):
    """Initialize parameters for an L-layer fully-connected network.

    Args:
        layers_dims: list of layer sizes [n0, n1, ..., nL] with nL == 1 for binary output
        scheme: one of {"bad", "xavier", "he"}
        activation: hidden-layer activation name ("tanh", "sigmoid", or "relu")
        seed: numpy RNG seed for reproducibility

    Returns:
        parameters: dict with keys {W1,b1,...,WL,bL}

    Notes:
        - "bad": deliberately poor init (Normal(0, 2.0^2), i.e. std = 2) to trigger saturation/vanishing with tanh/sigmoid
        - "xavier": Glorot uniform for tanh/sigmoid (limit = sqrt(6/(fan_in+fan_out)))
        - "he": Kaiming/He uniform for ReLU (limit = sqrt(6/fan_in))
        - Biases are initialized to zeros in all cases
    """
    np.random.seed(seed)
    L = len(layers_dims) - 1
    parameters = {}

    scheme = scheme.lower()
    activation = activation.lower()

    for l in range(1, L + 1):
        n_prev, n_curr = layers_dims[l-1], layers_dims[l]
        fan_in, fan_out = _fan_in_out(n_prev, n_curr)

        if scheme == "bad":
            # Large variance regardless of activation (to show failure with tanh/sigmoid)
            W = np.random.randn(n_curr, n_prev) * 2.0
        elif scheme == "xavier":
            # Recommended for tanh/sigmoid
            limit = np.sqrt(6.0 / (fan_in + fan_out))
            W = np.random.uniform(-limit, limit, size=(n_curr, n_prev))
        elif scheme == "he":
            # Recommended for ReLU
            limit = np.sqrt(6.0 / fan_in)
            W = np.random.uniform(-limit, limit, size=(n_curr, n_prev))
        else:
            raise ValueError("Unknown initialization scheme: " + scheme)

        b = np.zeros((n_curr, 1))
        parameters[f"W{l}"] = W
        parameters[f"b{l}"] = b

    return parameters


def predict(parameters, X, threshold=0.5, hidden_activation="tanh"):
    """Return binary predictions (0/1) using the current parameters and sigmoid head."""
    AL, _ = forward_propagation_deep(X, parameters, activation=hidden_activation)
    return (AL >= threshold).astype(int)


def accuracy(parameters, X, Y, hidden_activation="tanh", threshold=0.5):
    """Compute accuracy in [0,1]."""
    if Y.ndim == 1:
        Y = Y.reshape(1, -1)
    Y_pred = predict(parameters, X, threshold=threshold, hidden_activation=hidden_activation)
    return float(np.mean(Y_pred == Y))


# To do:

Analyse the following functions and for each function add a description.

In [ ]:
def plot_saturation_evolution(sat_hist, activation="tanh"):
    """
    Plot saturation fraction per layer over training.

    sat_hist: dict like { "A1": [..], "A2": [..], ... }

    Interpretation:
      - tanh: fraction of units with |A| > 0.99 (flat region, tanh' ~ 0 -> vanishing gradients)
      - sigmoid: fraction of units with A < 0.01 or A > 0.99 (sigmoid' ~ 0)
      - relu: fraction of units with A = 0 (inactive / dead ReLUs, zero gradient)
    """
    plt.figure(figsize=(8, 5))
    for key, series in sat_hist.items():
        if len(series) > 0:
            plt.plot(series, label=key)
    plt.xlabel("Iteration")
    plt.ylabel("Saturation fraction")
    plt.title(f"Saturation evolution per layer ({activation})")
    plt.ylim(0.0, 1.0)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_weights_evolution(initial_params, trained_params, activation="relu"):
    """
    Compare initial vs final weight distributions for each layer.

    Args:
        initial_params: dict {W1, b1, ..., WL, bL} BEFORE training
        trained_params: dict {W1, b1, ..., WL, bL} AFTER training
        activation: activation name (used only for plot titles)

    Behavior:
        - One subplot per layer (W1, W2, ..., WL)
        - Shows histograms (initial vs final) to visualize
          shrinkage, explosion, or stabilization of weights.
    """

    # Get only weight keys ("W1", "W2", ...)
    weight_keys = [k for k in trained_params.keys() if k.startswith("W")]
    n_layers = len(weight_keys)

    # Layout: 3 columns, enough rows
    cols = 3
    rows = int(np.ceil(n_layers / cols))

    plt.figure(figsize=(5 * cols, 3.5 * rows))

    for i, key in enumerate(weight_keys, start=1):
        w_init = initial_params[key].flatten()
        w_final = trained_params[key].flatten()

        plt.subplot(rows, cols, i)
        plt.hist(w_init, 50, alpha=0.5, label="initial")
        plt.hist(w_final, 50, alpha=0.5, label="final")
        plt.title(f"{key} — {activation}")
        plt.xlabel("Weight value")
        plt.ylabel("Frequency")
        plt.legend()

    plt.suptitle(f"Weight distributions evolution ({activation})", fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_gradients_evolution(grad_hist, activation="relu", metric="std", logy=True):
    """Plot gradient evolution per layer over iterations.

    Args:
        grad_hist: gradient history produced by init_grad_history/update_grad_history
        activation: name used only for figure titles/labels
        metric: same metric that was used to build the history (for labeling)
        logy: if True, use a log scale on the y-axis (helpful to reveal vanishing)

    Behavior:
        - One figure per tracked gradient kind (dZ, dW, db)
        - Each layer is a separate line in the figure
    """
    for kind, per_layer in grad_hist.items():
        plt.figure(figsize=(8, 5))
        for key, series in per_layer.items():
            if len(series) > 0:
                plt.plot(series, label=key)
        if logy:
            plt.yscale("log")
        plt.xlabel("Iteration")
        plt.ylabel(f"{kind} — {metric}")
        plt.title(f"Gradient evolution ({kind}) — {activation}")
        plt.legend(loc="best", fontsize=9, ncol=2)
        plt.tight_layout()
        plt.show()


# Task #1: Let's go through all the steps with one activation function, one initialization method and one dataset.

Activation function = Tanh

Initialization method = Bad

Dataset : circles

## T1.1: Load & preprocess the "circles" dataset

In [ ]:
# Choose dataset
DATASET = "circles"   # or "cats" or "moons"

USE_PCA = True     # True = faster training, clearer gradients
PCA_DIM = 100

if DATASET == "cats":
    X_train, y_train, X_test, y_test = load_dataset_flexible(
        dataset="cats",
        cats_train_path="data/train_catvnoncat.h5",
        cats_test_path="data/test_catvnoncat.h5",
        use_pca=USE_PCA,
        pca_dim=PCA_DIM,
    )
else:
    X_train, y_train, X_test, y_test = load_dataset_flexible(dataset=DATASET)

# Keep 2000 training points so that pure-NumPy training stays fast (test set unchanged)
X_train, y_train = X_train[:2000], y_train[:2000]

print("X_train:", X_train.shape, "X_test:", X_test.shape)

## T1.2: Define architecture 

In [ ]:
# last = 1 for binary classification (sigmoid)

n_x = X_train.shape[1]
layer_dims = [n_x, 64, 64, 64, 1]  # Should be deep enough to show vanishing...

print("Architecture:", layer_dims)

## T1.3: Train the model

In [ ]:
HIDDEN_ACT = "tanh"
INIT_SCHEME = "bad"

# The framework expects X with shape (n_features, m)
parameters, losses, grad_hist, initial_params, snapshots, sat_hist = train_deep_model(
    X_train.T, y_train, layer_dims, init_scheme=INIT_SCHEME, hidden_activation=HIDDEN_ACT,
    learning_rate=0.01, num_iterations=1000, print_every=200)

print("Train acc:", accuracy(parameters, X_train.T, y_train, HIDDEN_ACT),
      "| Test acc:", accuracy(parameters, X_test.T, y_test, HIDDEN_ACT))


## T1.4: Plot gradient evolution

In [ ]:
plot_gradients_evolution(grad_hist, activation=HIDDEN_ACT)

## T1.5: Plot saturation evolution

In [ ]:
plot_saturation_evolution(sat_hist, activation=HIDDEN_ACT)

## T1.6: Plot weight distribution

In [ ]:
plot_weights_evolution(initial_params, parameters, activation=HIDDEN_ACT)

### Your analysis here:
TODO_T1

# **Task #2**

The purpose of the following exercises is to help you **experiment**, **observe**, and **understand** how initialization and activation functions influence the learning dynamics of deep neural networks.

You have nothing to code — the framework is fully provided.
Your task is to **run experiments**, **interpret plots**, and **answer analysis questions**.

## **Exercise 1: Observing vanishing gradients on a 2D dataset**

### **Objective:**

Understand how **bad initialization** causes **activation saturation** and **vanishing gradients**, even on simple datasets.

### Instructions

1. Set dataset to `"circles"` or `"moons"`.
2. Use a deep architecture (e.g. `[2, 64, 64, 64, 64, 1]`).
3. Train with:

   * `hidden_activation="tanh"`
   * `init_scheme="bad"`
    
4. Plot:

   * `plot_gradients_evolution(...)`
   * `plot_saturation_evolution(...)`
   * `plot_weights_evolution(...)`

### **What to observe**

* Gradients shrinking layer by layer
* Activation saturation in middle/late layers
* Weight collapse during training

### **Questions**

1. Which layers are the first to vanish? Why?
2. What does the saturation curve tell you about tanh behavior?
3. Why does “bad initialization” make training impossible?

In [ ]:
# Helper reused in all exercises: train + accuracy + requested plots
def run_experiment(X, y, Xt, yt, dims, act, init, lr=0.01, iters=1000, plots=("grad", "sat", "w")):
    params, losses, grad_hist, init_params, _, sat_hist = train_deep_model(
        X.T, y, dims, init_scheme=init, hidden_activation=act,
        learning_rate=lr, num_iterations=iters, print_every=iters // 4)
    print(f"{act} + {init} (lr={lr}) | final loss = {losses[-1]:.4f} | "
          f"train acc = {accuracy(params, X.T, y, act):.3f} | test acc = {accuracy(params, Xt.T, yt, act):.3f}")
    if "loss" in plots:
        plt.plot(losses); plt.xlabel("Iteration"); plt.ylabel("Loss"); plt.title(f"Loss - {act} + {init}"); plt.show()
    if "grad" in plots:
        plot_gradients_evolution(grad_hist, activation=act)
    if "sat" in plots:
        plot_saturation_evolution(sat_hist, activation=act)
    if "w" in plots:
        if all(np.isfinite(v).all() for v in params.values()):
            plot_weights_evolution(init_params, params, activation=act)
        else:
            print("Final weights contain NaN/inf (diverged): histogram cannot be drawn.")
    return params, losses, grad_hist, init_params, sat_hist

DATASET = "circles"
X_train, y_train, X_test, y_test = load_dataset_flexible(dataset=DATASET)
X_train, y_train = X_train[:2000], y_train[:2000]
deep_dims = [2, 64, 64, 64, 64, 1]

res_tanh_bad = run_experiment(X_train, y_train, X_test, y_test, deep_dims, "tanh", "bad")


TODO_E1

## **Exercise 2: Fixing vanishing gradients with Xavier initialization**

### **Objective:**

Show empirically that **Xavier initialization** stabilizes tanh networks.

### Instructions

Repeat Exercise 1 but with:

* `init_scheme="xavier"`

### What to observe

* Gradients remain stable
* Saturation greatly reduced
* Training loss decreases properly

### **Questions**

1. Compare gradient curves between bad and Xavier. What changed?
2. Explain why Xavier’s variance formula solves the tanh saturation problem.
3. Does training become faster? Why?

In [ ]:
res_tanh_xavier = run_experiment(X_train, y_train, X_test, y_test, deep_dims, "tanh", "xavier")

TODO_E2

## **Exercise 3: Dead ReLU phenomenon with bad initialization**

### **Objective:**

Understand how large-variance initialization kills ReLU neurons.

### Instructions

1. Set dataset to `"circles"` or `"moons"`.
2. Use:

   * `hidden_activation="relu"`
   * `init_scheme="bad"`
3. Plot saturation (dead ReLUs = A=0).
4. Plot gradient evolution.

### What to observe

* Many neurons output exactly **0**
* Dead ReLUs become permanent
* Gradients drop to zero in deeper layers

### **Questions**

1. How does the saturation curve reveal dead ReLUs?
2. Why do gradients collapse faster with ReLU than with tanh?
3. What happens to the weight histograms?

In [ ]:
# Instructions: saturation (dead ReLUs) + gradients; weights are plotted only if they stay finite
res_relu_bad = run_experiment(X_train, y_train, X_test, y_test, deep_dims, "relu", "bad")
print("Saturation (dead fraction) of A1..A4, first 8 iterations:")
for k in ["A1", "A2", "A3", "A4"]:
    print(k, np.round(res_relu_bad[4][k][:8], 2))
print("Fraction of NaN in final W1:", np.isnan(res_relu_bad[0]["W1"]).mean())


TODO_E3

## **Exercise 4: Fix dead ReLUs with He initialization**

### **Objective:**

Show that He initialization preserves gradient flow in ReLU networks.

### Instructions

Repeat Exercise 3 but with:

* `init_scheme="he"`

### What to observe

* Almost no dead ReLUs
* Gradient curves stay active
* Loss decreases smoothly

### **Questions**

1. Compare the saturation curves between bad init and He.
2. What property of He initialization preserves gradient variance?
3. Does weight evolution look more stable?

In [ ]:
res_relu_he = run_experiment(X_train, y_train, X_test, y_test, deep_dims, "relu", "he")

TODO_E4

## **Exercise 5: Moving to a real dataset: Cats vs. non-cats**

### **Objective:**

Test whether the same initialization/activation principles apply to real images.

### Instructions

1. Set `DATASET = "cats"`
2. Try with `USE_PCA=True` and `USE_PCA=False`
3. Run three scenarios:

   * `tanh + bad`
   * `tanh + xavier`
   * `relu + he`
4. Plot:

   * gradients
   * saturation
   * weights
   * loss curve

### What to observe

* Bad initialization fails even more dramatically
* PCA helps stabilization
* Xavier/He still solve the problem

### **Questions**

1. How does PCA change the gradient evolution?
2. Why does `tanh + bad` perform much worse on images?
3. Which combination (activation + init) works best and why?

In [ ]:
cats_results = {}
for USE_PCA in [True, False]:
    print(f"\n================ USE_PCA = {USE_PCA} ================")
    Xc_train, yc_train, Xc_test, yc_test = load_dataset_flexible(
        dataset="cats",
        cats_train_path="data/train_catvnoncat.h5",
        cats_test_path="data/test_catvnoncat.h5",
        use_pca=USE_PCA,
        pca_dim=100,
    )
    cats_dims = [Xc_train.shape[1], 64, 64, 64, 64, 1]
    for act, init in [("tanh", "bad"), ("tanh", "xavier"), ("relu", "he")]:
        print(f"\n----- cats | PCA={USE_PCA} | {act} + {init} -----")
        cats_results[(USE_PCA, act, init)] = run_experiment(
            Xc_train, yc_train, Xc_test, yc_test, cats_dims, act, init,
            plots=("loss", "grad", "sat", "w"))


TODO_E5

## **Exercise 6: Depth Sensitivity**

### **Objective:**

Find the maximum depth at which each initialization still trains correctly.

### Instructions

Try architectures like:

* `[n_x, 32, 32, 32, 1]`
* `[n_x, 64, 64, 64, 64, 1]`
* `[n_x, 128, 128, 128, 128, 128, 1]`

For each depth, test:

* tanh + bad
* tanh + xavier
* relu + bad
* relu + he

### What to observe

* dramatic degradation as depth increases
* Xavier and He push the failure point further

### **Questions**

1. Which initialization degrades fastest with depth?
2. Which combination scales best with depth?
3. How do gradient curves differ at depth 3 vs depth 6?

In [ ]:
# circles (2000 train points), 500 iterations per run, lr = 0.01
archs = {3: [2, 32, 32, 32, 1], 4: [2, 64, 64, 64, 64, 1], 5: [2, 128, 128, 128, 128, 128, 1]}
combos = [("tanh", "bad"), ("tanh", "xavier"), ("relu", "bad"), ("relu", "he")]
depth_hist = {}
rows = []
with np.errstate(all="ignore"):
    for depth, dims in archs.items():
        for act, init in combos:
            print(f"\n----- {dims} | {act} + {init} -----")
            params, losses, gh, _, sh = run_experiment(X_train, y_train, X_test, y_test, dims, act, init,
                                                       iters=500, plots=())
            depth_hist[(depth, act, init)] = gh
            dW0 = [gh["dW"][k][0] for k in gh["dW"]]
            rows.append((depth, f"{act}+{init}", losses[-1], accuracy(params, X_test.T, y_test, act),
                         dW0[0] / dW0[-1], np.mean([sh[k][-1] for k in list(sh)[:-1]])))

print(f"\n{'hidden':>6} | {'combo':<12} | {'final loss':>10} | {'test acc':>8} | {'dW1/dWL (it 0)':>14} | {'mean hidden sat':>15}")
for r in rows:
    print(f"{r[0]:>6} | {r[1]:<12} | {r[2]:>10.4f} | {r[3]:>8.3f} | {r[4]:>14.2e} | {r[5]:>15.2f}")

# Gradient curves: shallow (3 hidden layers) vs deep (5 hidden layers) for tanh + bad and tanh + xavier
for key in [(3, "tanh", "bad"), (5, "tanh", "bad"), (3, "tanh", "xavier"), (5, "tanh", "xavier")]:
    print(key)
    plot_gradients_evolution({"dW": depth_hist[key]["dW"]}, activation=f"{key[1]}+{key[2]}, {key[0]} hidden")


TODO_E6

## **Exercise 7: Effect of learning rate on initialization**

### **Objective:**

Understand that the effect of initialization depends on learning rate.

### Instructions

Pick:

* `relu + he`
  Test learning rates:

* 0.001

* 0.01

* 0.1

Plot gradients and saturation.

### What to observe

* small LR → slow learning but stable
* large LR → exploding gradients or dead ReLUs

### **Questions**

1. Which LR gives the best gradient flow?
2. Why does too large LR interact badly with ReLU?
3. Does weight evolution reveal instability?

In [ ]:
lr_results = {}
for lr in [0.001, 0.01, 0.1]:
    print(f"\n----- relu + he | lr = {lr} -----")
    lr_results[lr] = run_experiment(X_train, y_train, X_test, y_test, deep_dims, "relu", "he", lr=lr,
                                    plots=("loss", "grad", "sat", "w"))


TODO_E7

## **Exercise 8: Weight evolution interpretation**

### **Objective:**

Connect weight evolution to stability of training.

### Instructions

For a selected experiment, compare:

* `plot_weights_evolution(initial_params, parameters)`
* `plot_gradients_evolution(grad_hist)`
* `plot_saturation_evolution(sat_hist)`

### What to observe

* With bad init → weights shrink or blow up
* With Xavier/He → distributions remain centered and stable

### **Questions**

1. How does weight variance correlate with gradient variance?
2. Which layers drift the most?
3. Does saturation match the behavior of weights?


In [ ]:
# Selected experiment: tanh + bad (Exercise 1), compared with tanh + xavier (Ex. 2) and relu + he (Ex. 4)
params, losses, grad_hist, init_params, sat_hist = res_tanh_bad
plot_weights_evolution(init_params, params, activation="tanh (bad)")
plot_gradients_evolution(grad_hist, activation="tanh (bad)")
plot_saturation_evolution(sat_hist, activation="tanh (bad)")

# Per-layer summary: weight std (init -> final), dW std (first -> last iteration), final saturation
for name, (p, _, gh, ip, sh) in [("tanh+bad", res_tanh_bad), ("tanh+xavier", res_tanh_xavier), ("relu+he", res_relu_he)]:
    print(f"\n{name}")
    for l in range(1, len(deep_dims)):
        print(f"  layer {l}: W std {ip[f'W{l}'].std():.3f} -> {p[f'W{l}'].std():.3f} | "
              f"dW std {gh['dW'][f'dW{l}'][0]:.2e} -> {gh['dW'][f'dW{l}'][-1]:.2e} | sat {sh[f'A{l}'][-1]:.2f}")


TODO_E8

## **Summary task: Final report**

### **Objective:**

Synthesize all your experimental findings.

### Write a short report answering:

1. Why do gradients vanish or explode in deep networks?
2. Why do poor initializations saturate tanh/sigmoid or kill ReLUs?
3. Why do Xavier and He initialization solve these problems?
4. How do toy 2D experiments compare to real image data?
5. How do gradient evolution, saturation, and weight evolution relate?

Your report here:

TODO_REPORT